# Testing Scraper on GDELT Data

### Imports and Notebook Settings


In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import os 
import re
from pathlib import Path

In [37]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
# import seaborn as sns 
import plotly.express as px
import pprint 

In [38]:
import requests
import trafilatura
from bs4 import BeautifulSoup
from datetime import datetime, timezone

In [39]:
ROOT = Path.cwd().parent

In [40]:
style_path = ( ROOT 
              / 'notebooks' 
              / 'styler.mplstyle'
              )
plt.style.use(style_path)

### Test scaping Methods

In [41]:
data_path = (ROOT 
             / 'data'
             / 'silver'
             / 'source=gdelt'
             /  'records.parquet'
             )

In [42]:
df = pd.read_parquet(data_path)

In [43]:
df = df.drop(columns=['raw'])

In [44]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata'],
      dtype='object')

In [45]:
for i in df['title'].head(30):
    print(i)

Roseville cancer center files for bankruptcy
Students see Xcel bills on the rise with inception of new carbon tax
Xcel franchise negotiations likely to drag into next year
Duke Energy Shuts Down Crystal River Nuclear Plant - Greenpeace
The Fate of Earth Day
Storms and nasty weather all over the planet for Christmas
Opinion: Are You a Scumbag? Take This Simple Test
Boeing board of directors elects new member
Duke Energy's acquisition of Piedmont to create a natural gas giant
Minneapolis-St. Paul Airport powers up Minnesota's largest solar energy project
World leaders in politics, business, science and arts to speak at 2016 Aspen Ideas Festival
Power Politics: Government Regulation of Solar Power – Utne
Four Sided Guy: Stranger Things Explained – The Demogorgon
Welcome to Dukeville – Meet the People Impacted by Duke Energy's Decision to Clean Up Coal Ash Pits on the Yadkin River
Four Sided Guy: The Original Dungeons & Dragons or the OGDND
Iowa's status as a renewable energy leader: How w

In [46]:
pprint.pprint(df.iloc[3])

record_id                                      20250210173000-217
source                                                      gdelt
source_type                                                   api
title           Duke Energy Shuts Down Crystal River Nuclear P...
text                                                             
published_at                            2013-02-05T00:00:00+00:00
retrieved_at                     2026-06-24T17:56:55.345301+00:00
url             https://www.greenpeace.org/usa/duke-energy-shu...
region                                                       USNC
categories      [ENV_COAL, WB_508_POWER_SYSTEMS, WB_507_ENERGY...
metadata        {'source_common_name': 'greenpeace.org', 'gdel...
Name: 3, dtype: object


In [47]:
import nest_asyncio
nest_asyncio.apply()


In [48]:
from datetime import datetime, timezone
import json

import aiohttp
import trafilatura
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


def categorize_status(status: int | None) -> str:
    if status == 403:
        return "forbidden"
    if status == 404:
        return "not_found"
    if status == 429:
        return "rate_limited"
    if status is not None and 500 <= status < 600:
        return "server_error"
    if status is not None:
        return "http_error"
    return "unknown"


async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    timeout: int = 15,
) -> dict:
    try:
        async with session.get(url, timeout=timeout) as response:
            text = await response.text(errors="ignore")

            if response.status >= 400:
                return {
                    "success": False,
                    "html": None,
                    "status_code": response.status,
                    "error_type": categorize_status(response.status),
                    "error_message": f"HTTP {response.status}",
                    "method": "aiohttp",
                }

            return {
                "success": True,
                "html": text,
                "status_code": response.status,
                "error_type": None,
                "error_message": None,
                "method": "aiohttp",
            }

    except TimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "timeout",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except aiohttp.ClientConnectionError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "connection_error",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "unknown",
            "error_message": str(exc),
            "method": "aiohttp",
        }


async def fetch_html_with_playwright(
    url: str,
    timeout: int = 30000,
) -> dict:
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)

            page = await browser.new_page(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/137.0.0.0 Safari/537.36"
                )
            )

            response = await page.goto(
                url,
                wait_until="networkidle",
                timeout=timeout,
            )

            html = await page.content()
            status_code = response.status if response else None

            await browser.close()

        return {
            "success": True,
            "html": html,
            "status_code": status_code,
            "error_type": None,
            "error_message": None,
            "method": "playwright",
        }

    except PlaywrightTimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_timeout",
            "error_message": str(exc),
            "method": "playwright",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_error",
            "error_message": str(exc),
            "method": "playwright",
        }


def extract_with_trafilatura(
    html: str,
    url: str,
    extractor: str,
) -> dict:
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=False,
    )

    if result is None:
        return {
            "success": False,
            "extractor": extractor,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "error_type": "parse_failed",
            "error_message": "trafilatura returned None",
        }

    parsed = json.loads(result)
    text = parsed.get("text", "") or ""

    return {
        "success": True,
        "extractor": extractor,
        "title": parsed.get("title"),
        "author": parsed.get("author"),
        "date": parsed.get("date"),
        "text": text,
        "error_type": None,
        "error_message": None,
    }


async def scrape_article(
    session: aiohttp.ClientSession,
    url: str,
) -> dict:
    retrieved_at = datetime.now(timezone.utc).isoformat()

    fetch_result = await fetch_html(session, url)

    # if (
    #     not fetch_result["success"]
    #     and fetch_result["error_type"] == "forbidden"
    # ):
    #     fetch_result = await fetch_html_with_playwright(url)

    if not fetch_result["success"]:
        return {
            "success": False,
            "url": url,
            "status_code": fetch_result["status_code"],
            "error_type": fetch_result["error_type"],
            "error_message": fetch_result["error_message"],
            "fetch_method": fetch_result["method"],
            "extractor": None,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "text_length": 0,
            "retrieved_at": retrieved_at,
        }

    extracted = extract_with_trafilatura(
        html=fetch_result["html"],
        url=url,
        extractor=f"trafilatura_after_{fetch_result['method']}",
    )

    return {
        "success": extracted["success"],
        "url": url,
        "status_code": fetch_result["status_code"],
        "error_type": extracted["error_type"],
        "error_message": extracted["error_message"],
        "fetch_method": fetch_result["method"],
        "extractor": extracted["extractor"],
        "title": extracted["title"],
        "author": extracted["author"],
        "date": extracted["date"],
        "text": extracted["text"],
        "text_length": len(extracted["text"]),
        "retrieved_at": retrieved_at,
    }

In [49]:
import pandas as pd
import aiohttp
import asyncio


async def scrape_urls(urls: list[str]) -> pd.DataFrame:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }

    async with aiohttp.ClientSession(headers=headers) as session:
        results = [
            await scrape_article(session, url)
            for url in urls
        ]

    return pd.DataFrame(results)



In [50]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata'],
      dtype='object')

In [51]:
urls = df["url"].dropna().head(100).tolist()

results_df = await scrape_urls(urls)

In [52]:
results_df[
    [
        "success",
        "url",
        "title",
        "date",
        "text_length",
        "extractor",
    ]
]

,success,url,title,date,text_length,extractor
0,True,https://www.twincities.com/2007/01/25/rosevill...,Roseville cancer center files for bankruptcy,2007-01-25,3381,trafilatura_after_aiohttp
1,True,https://www.cuindependent.com/2007/04/13/stude...,Students see Xcel bills on the rise with incep...,2007-04-13,4066,trafilatura_after_aiohttp
2,True,https://www.coloradodaily.com/2009/07/31/xcel-...,Xcel franchise negotiations likely to drag int...,2009-07-31,8719,trafilatura_after_aiohttp
3,True,https://www.greenpeace.org/usa/duke-energy-shu...,Duke Energy Shuts Down Crystal River Nuclear P...,2013-02-05,2930,trafilatura_after_aiohttp
4,True,https://www.newyorker.com/magazine/2013/04/15/...,The Fate of Earth Day,2013-04-15,19587,trafilatura_after_aiohttp
...,...,...,...,...,...,...
95,True,https://www.benzinga.com/insights/analyst-rati...,Analyst Ratings For CMS Energy - CMS Energy (N...,2024-11-01,4272,trafilatura_after_aiohttp
96,False,https://theenterpriseleader.com/2024/11/01/gug...,None,None,0,None
97,True,https://www.whio.com/news/local/major-utility-...,Major utility company looking to raise rates f...,2024-11-01,2208,trafilatura_after_aiohttp
98,False,https://www.csrwire.com/press_releases/813116-...,None,None,0,None


In [53]:
success_rate = results_df['success'].sum() / len(results_df)

print(f"{success_rate * 100}% of articles could be scraped")

65.0% of articles could be scraped


In [54]:
results_df["error_type"].value_counts(dropna=False)

error_type
None                65
not_found           13
server_error        10
rate_limited         5
parse_failed         3
forbidden            2
unknown              1
connection_error     1
Name: count, dtype: int64

In [55]:
with pd.option_context("display.max_colwidth", None):
    display(
        results_df.loc[
            ~results_df["success"],
            [
                "url",
                "status_code",
                "error_type",
                "error_message",
            ],
        ]
    )

,url,status_code,error_type,error_message
15,https://www.thegazette.com/news/iowas-status-as-a-renewable-energy-leader-how-we-got-here-and-whats-next/,404.0,not_found,HTTP 404
24,https://www.kvor.com/2022/04/01/class-action-suit-filed-in-marshall-fire-disaster/,404.0,not_found,HTTP 404
33,https://menafn.com/1108840254/Entergy-Employees-Recognized-For-Donating-Over-230000-To-St-Jude-Childrens-Research-Hospital,307.0,parse_failed,trafilatura returned None
35,https://finance.yahoo.com/news/vanguard-index-fund-once-decade-081200364.html,NaN,unknown,"400, message='Got more than 8190 bytes when reading: b""connect-src \'self\' wss://*.finance.yahoo.com/ https://*.cdn.yimg.com https://*.oath.com https://*.ya..."".', url='https://finance.yahoo.com/news/vanguard-index-fund-once-decade-081200364.html'"
40,https://www.mankatofreepress.com/news/business/alliant-energy-q3-earnings-snapshot/article_0d6df7d1-a37f-5c10-a9db-57a2abb42b5d.html,404.0,not_found,HTTP 404
44,https://menafn.com/1108842234/Business-Response-To-Hurricane-Helene-Among-Social-Impact-Sustainability-Topics-Explored-At-3BL-Network-Effect-Charlotte,307.0,parse_failed,trafilatura returned None
45,https://menafn.com/1108842237/Southern-Company-Leads-Energy-Innovation-With-Advanced-Technologies-And-RD-Investments-For-A-Clean-Energy-Future,307.0,parse_failed,trafilatura returned None
47,https://www.modernreaders.com/news/2024/11/01/alliant-energy-nasdaqlnt-updates-fy24-earnings-guidance.html,526.0,server_error,HTTP 526
48,https://www.wkrb13.com/2024/11/01/alliant-energy-nasdaqlnt-releases-fy25-earnings-guidance.html,526.0,server_error,HTTP 526
59,https://www.themarketsdaily.com/2024/11/01/catalyst-financial-partners-llc-has-310000-position-in-the-southern-company-nyseso.html,404.0,not_found,HTTP 404


In [56]:
results_df.columns

Index(['success', 'url', 'status_code', 'error_type', 'error_message',
       'fetch_method', 'extractor', 'title', 'author', 'date', 'text',
       'text_length', 'retrieved_at'],
      dtype='object')

In [57]:
import aiohttp
import asyncio
import json
import re

from bs4 import BeautifulSoup


TIMESTAMP_PATTERNS = {
    "second": re.compile(
        r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"
    ),
    "minute": re.compile(
        r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(?!:)"
    ),
    "date_only": re.compile(
        r"^\d{4}-\d{2}-\d{2}$"
    ),
}


def classify_timestamp(ts: str | None) -> str:
    if not ts:
        return "missing"

    if TIMESTAMP_PATTERNS["second"].search(ts):
        return "second"

    if TIMESTAMP_PATTERNS["minute"].search(ts):
        return "minute"

    if TIMESTAMP_PATTERNS["date_only"].search(ts):
        return "date_only"

    return "other"


def extract_timestamp(html: str) -> str | None:
    soup = BeautifulSoup(html, "html.parser")

    # OpenGraph
    meta = soup.find(
        "meta",
        attrs={"property": "article:published_time"}
    )

    if meta:
        return meta.get("content")

    # JSON-LD
    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        try:
            data = json.loads(script.string)

            if isinstance(data, dict):
                if "datePublished" in data:
                    return data["datePublished"]

            elif isinstance(data, list):
                for item in data:
                    if (
                        isinstance(item, dict)
                        and "datePublished" in item
                    ):
                        return item["datePublished"]

        except Exception:
            pass

    return None


async def analyze_urls(urls):
    stats = {
        "second": 0,
        "minute": 0,
        "date_only": 0,
        "other": 0,
        "missing": 0,
        "failed": 0,
    }

    examples = []

    async with aiohttp.ClientSession() as session:

        for i, url in enumerate(urls):
            try:
                fetch_result = await fetch_html(
                    session,
                    url,
                )

                if not fetch_result["success"]:
                    stats["failed"] += 1
                    continue

                ts = extract_timestamp(
                    fetch_result["html"]
                )

                category = classify_timestamp(ts)

                stats[category] += 1

                if len(examples) < 20:
                    examples.append(
                        {
                            "url": url,
                            "timestamp": ts,
                            "category": category,
                        }
                    )

            except Exception:
                stats["failed"] += 1

            if i % 100 == 0:
                print(f"Processed {i}")

    print("\nResults")
    print("-" * 50)

    total = len(urls)

    for k, v in stats.items():
        pct = 100 * v / total
        print(f"{k:12s}: {v:6d} ({pct:.1f}%)")

    return stats, examples

In [62]:
df['metadata'].iloc[0]

{'source_common_name': 'twincities.com',
 'gdelt_timestamp': '20250409144500',
 'gdelt_date': '20250409144500',
 'page_precise_pub_timestamp': '20070125000000',
 'has_precise_published_at': True,
 'published_at_source': 'page_precise_pub_timestamp',
 'published_at_precision': 'second',
 'organizations': array(['xcel energy', 'parker hughes cancer center'], dtype=object),
 'persons': array(['robert spande', 'nancy dvorak', 'fatih uckun', 'parker hughe',
        'jeremy olson', 'kenneth corey-edstrom', 'parker hughes'],
       dtype=object),
 'locations': array(['US-USIL', 'USMN', 'USIL', 'US', 'US-USMN'], dtype=object),
 'tone': '-3.36134453781513,2.01680672268908,5.3781512605042,7.39495798319328,22.3529411764706,0.840336134453782,513',
 'theme_match': False,
 'organization_match': True,
 'location_match': True,
 'filter_match_count': None}

In [69]:
df["has_precise_published_at"] = df["metadata"].apply(
    lambda x: x.get("has_precise_published_at", False)
)

records_without_precise = df.loc[~df['has_precise_published_at'], 'url']

In [75]:
stats, examples = await analyze_urls(records_without_precise[:500])

CancelledError: 

In [76]:
stats

{'second': 250,
 'minute': 0,
 'date_only': 3,
 'other': 2,
 'missing': 41,
 'failed': 204}